# Chapter 3 - Lab 1: LangChain agent

**Goal**: build a customer-experience analysis agent with **LangChain**'s `create_agent`, which wires together two tools, a system prompt and an LLM that decides which tool to call and when.

This is the same reference task used across every framework lab in Chapter 3:
**compare the Zenith X1 smartphone (`ZX1`) and the FrostLine refrigerator (`FL-340`) on contact rate**
(support tickets per 100 delivered orders) and write a short memo for the Head of Customer Care.
Comparing all frameworks on the *same* task makes differences in API style, abstractions and ergonomics easy to spot.

*Finance edition*: compare Apple and JPMorgan on P/E. Contact rate plays the role of P/E: a ratio
that lets a high-volume and a low-volume product be compared fairly (ZX1 = 15.0, FL-340 = 36.0).

**Tech stack**:
* **LangChain** (`langchain`) - `create_agent` + tool schemas inferred from type hints.
* **OpenAI** `gpt-4o-mini`.

## 1. Install packages

In [ ]:
%pip install -q langchain langchain-openai pydantic python-dotenv

## 2. Set up the API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared task setup

The task, tools, dataset and prompts live in `common.py` (next to this notebook in the repository).

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%203/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
from common import (
    get_support_data,
    compute_contact_rate,
    support_data,
    system_message,
    input_message,
)

print("Tools loaded. Reference task:")
print(" ", input_message)

## 4. Create the LangChain agent

`create_agent` accepts a list of tools, a `provider:model` string and a system prompt. LangChain infers each tool's schema from its signature and docstring.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    tools=[get_support_data, compute_contact_rate],
    model="openai:gpt-4o-mini",
    system_prompt=system_message,
)

## 5. Run the agent

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": input_message}]})

## 6. Inspect the agent's trace

In [ ]:
for msg in result.get("messages", []):
    msg_type = type(msg).__name__
    if msg_type == "ToolMessage":
        print(f"[Tool: {msg.name}]\n{msg.content}\n")
    elif msg_type == "AIMessage" and msg.content:
        print(f"[Agent Response]\n{msg.content}\n")

## 7. Results
Expect `get_support_data` once per product line, `compute_contact_rate` for each, then the memo
(ZX1 = 15.0, FL-340 = 36.0 tickets per 100 orders: the refrigerator line generates 2.4x more support load per order).

**What to notice about LangChain**: switching provider is a one-line change (`"anthropic:..."`);
the returned message list (`AIMessage`, `ToolMessage`) makes the trace easy to audit.